# QML 강의 - 3일차

## 실습: MediQ-GAN 학습 + QGAN과의 비교

**목표**: 2일차에 학습시킨 QGAN(생성자 전체가 양자)과, 오늘 다루는 MediQ-GAN(생성자 내부만 일부 양자)을
**같은 지표로 나란히 비교함.** MediQ-GAN은 원 논문 스펙(16큐빗 x 5개 sub-generator, depth 8)을 그대로 쓰되,
사전학습해둔 checkpoint에서 출발해 수업 중 일부 step만 추가 학습(미세조정)함.

**왜 전체 학습을 하지 않는가**: 16큐빗 회로는 상태벡터 시뮬레이션 자체가 비싸, 샘플 1개 1 step에 약 3초가 걸림.
DF 115장 기준 1 epoch이 약 8분이라 25 epoch에 196분(약 3시간 15분)이 걸렸음(사전 수행함). 

**실습 자산**: 강의 GitHub 저장소(https://github.com/hyunhp/skin_disease_qml)의 `practice/assets/` 폴더.
1번 섹션에서 `git clone`으로 받아 사용함(로그인 불필요, 수 MB).

**목차**:
2일차 복습 + QGAN 생성·평가(20분) → MediQ-GAN 구조(15분) → 라이브 미세조정(25분) → 두 모델 비교(20분) → 질의 응답(10분)

## 0. Google Colab 사용법 (빠른 복습)

2일차와 동일하게 **Google Colab**에서 진행함. 2일차에 참석하지 못했거나 Colab이 처음이면 2일차 노트북 **"0. Google Colab 시작하기"**(계정 만들기 포함)를 먼저 확인함.

### 0-1. 시작 전 확인
- Google 계정으로 https://colab.research.google.com 에 로그인되어 있는지 확인함 (학교·회사 계정이 막혀 있으면 개인 Gmail 사용).
- 노트북을 연 뒤 **`파일(File) → Drive에 사본 저장(Save a copy in Drive)`** 실행 — 사본을 만들어야 실행 결과가 저장됨.
- "이 노트북은 Google에서 작성하지 않았습니다(This notebook was not authored by Google)" 경고가 뜨면 `무시하고 계속(Run anyway)` 클릭.
- **런타임은 기본 CPU로 충분함** — GPU로 바꿀 필요 없음(양자회로 시뮬레이션은 CPU로 실행되며, GPU로 빨라지지 않음).

### 0-2. 자주 쓰는 조작
| 하고 싶은 것 | 방법 |
|---|---|
| 셀 실행 | **Shift + Enter**, 또는 셀 왼쪽 ▶ 버튼 |
| 실행 중단 | 셀 왼쪽 ■ 버튼, 또는 `런타임(Runtime) → 실행 중단(Interrupt execution)` |
| 처음부터 전부 실행 | `런타임(Runtime) → 모두 실행(Run all)` |
| 꼬였을 때 초기화 | `런타임(Runtime) → 세션 다시 시작(Restart session)` 후 1번 섹션부터 다시 실행 |

### 0-3. 오늘 실습에서 알아둘 점
- **2일차 런타임은 이미 종료된 상태임** — 2일차에 설치한 라이브러리·변수는 남아 있지 않으므로 오늘도 1번 섹션 설치 셀부터 **위에서 아래로 순서대로** 실행함(건너뛰면 `NameError` 발생).
- **로그인이 필요 없음** — 실습 자산은 공개 GitHub 저장소에서 `git clone`으로 받으며, 수 MB라 수 초면 끝남(원본 6GB는 받지 않음).
- 오늘은 사전학습 checkpoint를 불러와 쓰므로 대부분의 셀이 수 초 안에 끝남. 시간이 걸리는 셀은 두 개임:
  - 3번 섹션 MediQ-GAN 구조 확인(16큐빗 회로 5개 forward 1회) - 십여 초
  - 4번 섹션 미세조정 20 step - 로컬 CPU 실측 약 7분, Colab 무료 CPU는 더 걸릴 수 있음
- 수업 도중 연결이 끊기면(유휴 상태가 길 때 발생) 다시 연결한 뒤 1번 섹션부터 다시 실행하면 됨 — 미세조정 셀을 제외하면 1 ~ 2분 이내임.
- 오류가 나면 **오류 메시지 마지막 줄**을 먼저 확인함 — 대부분 앞 셀 미실행 또는 연결 끊김이 원인임.

## 1. 2일차 복습 + 실습 자산 로드 (20분)

2일차에 만든 sub-generator 회로를 그대로 다시 구성함. 오늘 쓰는 checkpoint도 2일차와 **같은 스펙**임:
데이터 6 + ancilla 1 = 7큐빗, patch 64픽셀, sub-generator 4개, depth 10 (4 x 2^6 = 256 = 16x16).

차이는 학습량뿐임 - 2일차에는 수업 중 400 step을 돌렸고, 오늘 쓰는 강사 checkpoint는 600 step까지 학습한 것임.

**측정 방식(2일차와 동일)**: ancilla qubit까지 측정해 **ancilla qubit이 0인 경우의 확률만** patch로 사용함
(논문의 post-selection). patch마다 밝기 총합이 0 ~ 1 사이에서 학습되므로, 배경이 많은 patch와
병변이 있는 patch를 다르게 표현할 수 있음.

In [ ]:
!pip install -q pennylane pennylane-lightning koreanize-matplotlib
import os, math, json, time
import numpy as np, torch, torch.nn as nn
import pennylane as qml
import matplotlib.pyplot as plt
# 그래프 한글 표시 — Colab 기본 폰트에는 한글이 없어 설정하지 않으면 그래프 글자가 네모(□)로 깨짐
try:
    import koreanize_matplotlib  # 나눔고딕 폰트 등록 + 기본 폰트 지정
except Exception as e:
    print("한글 폰트 설정 실패 — 그래프의 한글만 깨지고 실습 진행에는 지장 없음:", e)

# 강의 저장소 — 실습 자산(2일차 전처리 데이터 + 사전학습 checkpoint)이 practice/assets/에 들어 있음
REPO_URL = "https://github.com/hyunhp/skin_disease_qml.git"
REPO_DIR = "/content/skin_disease_qml"

In [ ]:
# 저장소 내려받기 (이미 받아둔 경우에는 최신 상태로 갱신만 함)
import sys, subprocess

if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "-q"], check=False)
else:
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)

ASSET_DIR = f"{REPO_DIR}/practice/assets"
sys.path.insert(0, f"{REPO_DIR}/practice")  # gen_metrics.py를 import할 수 있게 경로 추가
import gen_metrics as gm          # 2일차에서 쓴 생성품질 지표 모듈과 동일함

for need in ["qgan_df_16px.pt", "mediqgan_e010.pt", "mediqgan_final.pt", "df_preprocessed.npy"]:
    if not os.path.exists(f"{ASSET_DIR}/{need}"):
        raise RuntimeError(f"실습 자산 {need}이 없음 - git clone 출력의 오류 메시지를 확인하고 이 셀을 다시 실행할 것")
print(f"실습 자산 경로: {ASSET_DIR}")
print(f"포함 항목: {sorted(os.listdir(ASSET_DIR))}")

df_imgs = np.load(f"{ASSET_DIR}/df_preprocessed.npy")
nv_imgs = np.load(f"{ASSET_DIR}/nv_preprocessed.npy")
IMG_SIZE = df_imgs.shape[1]  # 자산 해상도(16)를 그대로 사용 - 2일차 IMG_SIZE와 같아야 함
print(f"2일차 전처리 데이터 로드: DF {df_imgs.shape}, NV {nv_imgs.shape}, IMG_SIZE={IMG_SIZE}")

# (선택) 2일차에 본인이 Drive에 저장한 checkpoint로 진행하려면 아래 주석을 해제함
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
class PatchQuantumGenerator(nn.Module):
    """2일차와 같은 patch 방식 생성자 - sub-generator N개가 patch를 하나씩 만들어 이어붙임.

    인자 기본값은 아래에서 checkpoint config로 항상 덮어쓰므로 의미 없음."""
    def __init__(self, n_generators=4, n_data_qubits=4, n_ancillas=1, q_depth=3):
        super().__init__()
        self.n_generators = n_generators
        self.n_data_qubits = n_data_qubits
        self.n_qubits = n_data_qubits + n_ancillas
        self.q_depth = q_depth
        self.patch_dim = 2 ** n_data_qubits  # patch 하나의 픽셀 수
        self.anc_stride = 2 ** n_ancillas  # ancilla qubit이 마지막 wire → 'ancilla qubit=0' 항목은 이 간격마다 위치함

        dev = qml.device("default.qubit", wires=self.n_qubits)  # backprop 미분에는 lightning.qubit이 아니라 default.qubit
        n_qubits = self.n_qubits

        @qml.qnode(dev, interface="torch", diff_method="backprop")
        def circuit(noise, weights):
            for i in range(n_qubits):
                qml.RY(noise[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])
            # ancilla qubit까지 전체 측정 → forward에서 'ancilla qubit=0' 항목만 patch로 사용 (논문의 post-selection에 해당)
            return qml.probs(wires=range(n_qubits))

        self.circuit = circuit
        # sub-generator마다 독립된 회전각 (depth x 큐빗)
        self.q_params = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, self.n_qubits) * math.pi) for _ in range(n_generators)
        ])

    def forward(self, batch_size, device="cpu"):
        # sub-generator별로 batch_size장의 patch를 만든 뒤 가로로 이어붙임 → (batch, 픽셀 수)
        all_patches = []
        for gen_idx in range(self.n_generators):
            weights = self.q_params[gen_idx]
            # [::anc_stride] = ancilla qubit이 0인 경우의 확률만 선택 → patch 밝기 총합(0~1)도 학습 대상이 됨
            batch_patches = [self.circuit(torch.rand(self.n_qubits, device=device) * math.pi, weights)[::self.anc_stride]
                              for _ in range(batch_size)]
            all_patches.append(torch.stack(batch_patches))
        return torch.cat(all_patches, dim=1).float()

    @property
    def output_dim(self):
        return self.n_generators * self.patch_dim

print("PatchQuantumGenerator 클래스 정의 완료 — 아직 랜덤 초기화 상태(학습 전)임")

## 2. QGAN 생성 + 지표 4종 (2일차와 동일 잣대)

checkpoint(600 step 학습)를 불러와 이미지를 생성하고, 2일차에 쓴 지표 4종으로 평가함.
이 숫자가 뒤에서 MediQ-GAN과 비교할 **기준선**이 됨.

In [ ]:
ckpt_path = f"{ASSET_DIR}/qgan_df_{IMG_SIZE}px.pt"
checkpoint = torch.load(ckpt_path, map_location="cpu")  # GPU에서 저장했어도 CPU로 불러옴
cfg = checkpoint["config"]
print(f"불러온 checkpoint 설정: {cfg}")

gen = PatchQuantumGenerator(n_generators=cfg["n_generators"], n_data_qubits=cfg["n_data_qubits"],
                             n_ancillas=cfg["n_ancillas"], q_depth=cfg["q_depth"])
gen.load_state_dict(checkpoint["generator_state_dict"])
gen.eval()  # 평가 모드 (이 모델은 dropout·batchnorm이 없어 동작 차이는 없음)
print(f"학습 완료된 생성자 로드됨 (2일차에 각자 학습시킨 것과 같은 구조, step 수만 더 많음)")

In [ ]:
# QGAN 생성 (추론만 수행)
# 지표 표본 수. QGAN은 생성이 싸서 64장, MediQ-GAN은 16큐빗 시뮬레이션이 비싸 16장으로 둠
# (8~16장으로는 QGAN 평균 상관계수가 0.05~0.17까지 흔들렸음 - 소수점보다 두 모델의 격차를 볼 것)
N_EVAL_QGAN = 64
N_EVAL = 16
with torch.no_grad():
    qgan_fake = gen(batch_size=N_EVAL_QGAN).numpy().reshape(N_EVAL_QGAN, IMG_SIZE, IMG_SIZE)

qgan_u = gm.to_unit_scale(qgan_fake, "prob")   # 확률 척도 → 이미지별 min-max로 0~1
real_u = gm.to_unit_scale(df_imgs, "unit")      # 이미 0~1 - 그대로

fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for j in range(4):
    axes[0, j].imshow(qgan_u[j], cmap="gray"); axes[0, j].set_title(f"QGAN 생성 {j}", fontsize=9); axes[0, j].axis("off")
    axes[1, j].imshow(real_u[j], cmap="gray"); axes[1, j].set_title(f"실제 DF {j}", fontsize=9); axes[1, j].axis("off")
plt.suptitle("QGAN 생성 이미지(위) vs 실제 DF(아래)")
plt.tight_layout(); plt.show()

# 지표 4종을 한 번에 계산 - 5번 섹션에서 MediQ-GAN과 같은 함수로 비교함
qgan_scores = gm.evaluate(qgan_fake, df_imgs, "prob")
for k, v in qgan_scores.items():
    print(f"  {k}: {v:.4f}")

## 3. MediQ-GAN 구조 (15분)

MediQ-GAN은 생성자 **내부**에서 고전 stream과 양자 stream이 병렬로 돌고 skip-connection으로 합쳐지는
**layer 단위 hybrid**임. QGAN처럼 생성자 전체가 양자인 **모델 단위 hybrid**와 근본적으로 다름.

공식 레포(github.com/QingyueJ-nd/MediQ-GAN)와 같은 방식으로 구현함:
- 양자 쪽 입력 절반을 4x4 = 16 site로 보고, 그중 **5개 site만** 실제 양자회로가 처리함(나머지 11칸은 0)
- 회로: RY(x) → RX(x) encoding → [학습 RY + 인접 CZ] x 8층 → PauliX 기댓값 → 큐빗 평균
- 1x1 conv로 투영 후 고전 stream과 concat → decoder

**구현 세부** (다음 셀 `ToyDualStreamGenerator`, 공식 레포 `mediq-gan.py`의 `quantum_circuit`·`_quantum_branch_8to4`와 같은 방식)

| 단계 | 내용 |
|---|---|
| 입력 분할 | 잠재벡터를 절반씩 나눠 고전 stream과 양자 stream에 분배 (논문 Fig.2 "feature map split") |
| site 구성 | 양자 쪽 절반을 4×4 = 16 site로 보고, site별 벡터를 `q_align`(Linear)으로 16차원 토큰으로 맞춤 |
| site 선택 | `linspace(0, 15, 5).round()`로 고른 site [0, 4, 8, 11, 15]를 sub-generator마다 하나씩 입력 |
| 회로 | 같은 값으로 RY → RX encoding, [학습 RY + 인접 CZ] × 8층, 큐빗별 PauliX 기댓값 측정 |
| 출력 정리 | 큐빗 방향 평균 → site당 스칼라 1개 → 4×4 맵의 해당 칸에 배치(나머지 11칸은 0) |
| 투영·융합 | `proj_qexp`(1×1 conv, 칸별 독립)로 5채널 → 16채널 투영 후 고전 stream 출력과 concat → decoder → Tanh |

depth는 논문 그림 기준 8을 씀(공식 레포 CLI 기본값은 6).

**단순화한 부분**: 고전 encoder/decoder가 conv가 아니라 MLP이고, 입력이 64x64 RGB가 아니라 2일차 pipeline이
만든 16x16 저해상도 벡터임. 그리고 **판별자가 원 논문의 WGAN-GP critic이 아니라 QGAN과 같은 표준 BCE 판별자**임.
**양자 stream 자체는 원 논문 스펙(16큐빗 x 5, depth 8) 그대로임.**

In [ ]:
class ToyDualStreamGenerator(nn.Module):
    """MediQ-GAN dual-stream 생성자 (구조 설명은 위 마크다운 표 참고)."""

    def __init__(self, in_dim=64, n_qubits=16, q_depth=8, n_generators=5, out_dim=64,
                 q_ch=16, dev_name="lightning.qubit"):
        super().__init__()
        assert in_dim % 2 == 0, "고전/양자 stream으로 절반씩 나누려면 in_dim이 짝수여야 함"
        self.half = in_dim // 2
        assert self.half % 16 == 0, "양자 쪽 절반을 4×4=16 site로 나누려면 in_dim/2가 16의 배수여야 함"
        self.site_dim = self.half // 16
        self.n_qubits = n_qubits
        self.q_depth = q_depth
        self.n_generators = n_generators
        self.site_idx = torch.linspace(0, 15, steps=n_generators).round().long().tolist()  # [0, 4, 8, 11, 15]

        # 고전 stream — MediQ-GAN의 "classical encoder" 역할(단순화: conv 대신 MLP)
        self.classical_stream = nn.Sequential(
            nn.Linear(self.half, 32), nn.ReLU(),
            nn.Linear(32, self.half),
        )

        # 양자 stream — site별 벡터를 큐빗 수에 맞추는 aligner(레포의 q_align)
        self.q_align = nn.Linear(self.site_dim, n_qubits)

        dev = qml.device(dev_name, wires=n_qubits)

        # lightning.qubit(C++ 시뮬레이터)은 backprop을 지원하지 않아 adjoint 미분 사용 - 16큐빗에서 메모리·속도 모두 유리
        @qml.qnode(dev, interface="torch", diff_method="adjoint")
        def q_circuit(x, weights):
            # 공식 레포와 동일: 같은 값으로 RY→RX encoding, 학습 RY + 인접 CZ 반복, X-basis 측정
            for i in range(n_qubits):
                qml.RY(x[i], wires=i)
                qml.RX(x[i], wires=i)
            for layer in range(q_depth):
                for i in range(n_qubits):
                    qml.RY(weights[layer, i], wires=i)
                for i in range(n_qubits - 1):
                    qml.CZ(wires=[i, i + 1])
            return [qml.expval(qml.PauliX(i)) for i in range(n_qubits)]

        self.q_circuit = q_circuit
        self.q_weights = nn.ParameterList([
            nn.Parameter(torch.rand(q_depth, n_qubits) * math.pi) for _ in range(n_generators)
        ])
        # 레포의 proj_qexp: 5개 스칼라 맵(4×4) → q_ch채널, 1×1 conv라 칸별 독립 처리
        self.proj_qexp = nn.Conv2d(n_generators, q_ch, 1)

        # 융합 후 decoder — skip-connection으로 합쳐진 특징을 최종 출력으로 (MediQ-GAN의 decoder conv에 해당)
        fused_dim = self.half + q_ch * 16
        self.decoder = nn.Sequential(
            nn.Linear(fused_dim, 32), nn.ReLU(),
            nn.Linear(32, out_dim), nn.Tanh(),
        )

    def quantum_stream(self, z_quantum):
        # sub-generator 5개 x batch B개만큼 16큐빗 회로를 실행 - 이 모델에서 가장 느린 부분
        B = z_quantum.shape[0]
        sites = z_quantum.view(B, 16, self.site_dim)       # 4×4 = 16 site
        tokens = self.q_align(sites)                        # (B, 16, n_qubits)
        q_maps = torch.zeros(B, self.n_generators, 4, 4, dtype=z_quantum.dtype)
        for g, si in enumerate(self.site_idx):              # sub-generator g ↔ site si
            outs = torch.stack([torch.stack(self.q_circuit(tokens[b, si], self.q_weights[g]))
                                for b in range(B)])         # (B, n_qubits)
            y, x = divmod(si, 4)
            q_maps[:, g, y, x] = outs.mean(dim=1).to(z_quantum.dtype)  # 큐빗 평균 → 스칼라
        return self.proj_qexp(q_maps).flatten(1)            # (B, q_ch*16)

    def forward(self, z):
        # z: (batch, in_dim) — 잠재벡터를 절반씩 stream에 분배 (MediQ-GAN Fig.2: "feature map split")
        z_classical, z_quantum = z[:, :self.half], z[:, self.half:]
        classical_feat = self.classical_stream(z_classical)  # (batch, half)
        quantum_feat = self.quantum_stream(z_quantum)        # (batch, q_ch*16)
        fused = torch.cat([classical_feat, quantum_feat], dim=1)  # skip-connection 융합
        return self.decoder(fused)

toy_gen = ToyDualStreamGenerator(in_dim=IMG_SIZE*IMG_SIZE, n_qubits=16, q_depth=8, n_generators=5, out_dim=IMG_SIZE*IMG_SIZE)
z = torch.rand(1, IMG_SIZE*IMG_SIZE)  # 구조 확인용 입력 1개
t0 = time.time()
demo_out = toy_gen(z)
print(f"dual-stream 융합 출력 shape: {demo_out.shape} (forward {time.time()-t0:.1f}초) — 고전 stream + 양자 stream(16큐빗×5개 → site별 스칼라 → 1×1 conv)이 하나로 융합됨")
print("(학습 전 랜덤 가중치이므로 의미있는 이미지는 아님 — 구조 확인용. 양자회로는 공식 레포와 동일: RY→RX encoding, 학습 RY+CZ, PauliX 측정)")

## 4. 라이브 미세조정 (25분)

**덜 학습된 checkpoint**(`mediqgan_e010.pt`, 10 epoch)에서 출발해 수업 중 추가 학습을 돌림.
완전히 수렴한 모델에서 시작하면 손실이 거의 움직이지 않아 학습 과정이 보이지 않기 때문임.

판별자는 QGAN과 같은 논문 Table 4 구조(5층 완전연결)를 씀. **척도 주의**: MediQ-GAN 출력은 Tanh(-1 ~ 1)이므로
실제 이미지도 `2x-1`로 옮겨서 학습하고, 눈으로 보거나 지표를 계산할 때 `to_unit_scale(x, "tanh")`로 0 ~ 1로 되돌림.

In [ ]:
import torch.nn as nn

IN_DIM = IMG_SIZE * IMG_SIZE
# 10 epoch까지 학습한 checkpoint - 덜 학습된 상태에서 출발해야 미세조정 효과가 보임
mq_ckpt = torch.load(f"{ASSET_DIR}/mediqgan_e010.pt", map_location="cpu")
mq_cfg = mq_ckpt["config"]
print(f"MediQ-GAN checkpoint 설정: {mq_cfg}")

mq_gen = ToyDualStreamGenerator(in_dim=IN_DIM, n_qubits=mq_cfg["n_qubits"], q_depth=mq_cfg["q_depth"],
                                n_generators=mq_cfg["n_generators"], out_dim=IN_DIM)
mq_gen.load_state_dict(mq_ckpt["generator_state_dict"])

class ClassicalDiscriminator(nn.Module):
    """QGAN과 동일한 논문 Table 4 판별자."""
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, 16), nn.ReLU(),
            nn.Linear(16, 8), nn.ReLU(),
            nn.Linear(8, 1), nn.Sigmoid(),
        )
    def forward(self, x):
        return self.net(x)

mq_disc = ClassicalDiscriminator(IN_DIM)
mq_disc.load_state_dict(mq_ckpt["discriminator_state_dict"])  # 판별자도 같은 시점 상태로 불러와야 균형이 유지됨

# 잠재벡터 분포는 checkpoint 설정을 그대로 따름(강사 학습과 같은 조건이어야 비교가 됨)
Z_DIST = mq_cfg.get("z_dist", "normal")

def sample_z(n):
    return torch.rand(n, IN_DIM) if Z_DIST == "uniform" else torch.randn(n, IN_DIM)

print(f"잠재벡터 분포: {Z_DIST}")

# 미세조정 전 생성 결과를 먼저 저장해둠 (나중에 전후 비교)
with torch.no_grad():
    before = mq_gen(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)
print(f"미세조정 전 생성 완료 - 값 범위 {before.min():.3f}~{before.max():.3f} (Tanh이므로 -1~1이 정상)")

In [ ]:
import time

# 실제 이미지를 Tanh 척도로 (0~1 -> -1~1)
real_tanh = torch.tensor(df_imgs.reshape(len(df_imgs), -1) * 2 - 1, dtype=torch.float32)

bce = nn.BCELoss()
# betas도 강사 학습과 동일하게(GAN 관례 0.5) - 기본값 0.9로 두면 불안정함
_betas = (mq_cfg.get("beta1", 0.5), 0.999)
opt_g = torch.optim.Adam(mq_gen.parameters(), lr=mq_cfg["lr_g"], betas=_betas)
opt_d = torch.optim.Adam(mq_disc.parameters(), lr=mq_cfg["lr_d"], betas=_betas)

FINE_TUNE_STEPS = 20   # 실측 1 step 약 20초(batch 4 x 샘플당 약 3초, 판별자+생성자). Colab 무료 CPU는 더 느릴 수 있음
                       # 아래 출력의 '남은 예상'을 보고 오래 걸리면 중단 후 값을 줄여도 됨(10 step으로도 변화는 보임)
BATCH = 4
ft_g, ft_d = [], []

t0 = time.time()
for step in range(FINE_TUNE_STEPS):
    # 실제 DF에서 BATCH장 무작위 추출(복원추출)
    batch = real_tanh[torch.randint(0, len(real_tanh), (BATCH,))]
    real_lbl, fake_lbl = torch.ones(BATCH, 1), torch.zeros(BATCH, 1)

    # 판별자: 실제→1, 생성→0. detach로 생성자 쪽 기울기는 계산하지 않음
    opt_d.zero_grad()
    loss_d = bce(mq_disc(batch), real_lbl) + bce(mq_disc(mq_gen(sample_z(BATCH)).detach()), fake_lbl)
    loss_d.backward(); opt_d.step()

    # 생성자: 판별자가 생성 이미지를 1(진짜)로 판정하도록 학습 - 양자 stream 파라미터까지 역전파됨
    opt_g.zero_grad()
    loss_g = bce(mq_disc(mq_gen(sample_z(BATCH))), real_lbl)
    loss_g.backward(); opt_g.step()

    ft_g.append(loss_g.item()); ft_d.append(loss_d.item())
    el = time.time() - t0
    print(f"step {step+1}/{FINE_TUNE_STEPS}  loss_g={loss_g.item():.4f}  loss_d={loss_d.item():.4f}  "
          f"경과 {el/60:.1f}분  남은 예상 {(FINE_TUNE_STEPS-step-1)*el/(step+1)/60:.1f}분", flush=True)

print(f"\n미세조정 완료 - 총 {(time.time()-t0)/60:.1f}분")
print("양자 stream(sub-generator 5개 x 16큐빗) 파라미터까지 gradient가 전달되어 함께 갱신됨")

In [ ]:
# 미세조정 전후 비교 + 강사가 25 epoch까지 끝낸 최종본
with torch.no_grad():
    after = mq_gen(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)

# 강사가 로컬에서 25 epoch까지 학습한 최종본 (README 결과표의 MediQ-GAN 수치)
mq_final = ToyDualStreamGenerator(in_dim=IN_DIM, n_qubits=mq_cfg["n_qubits"], q_depth=mq_cfg["q_depth"],
                                  n_generators=mq_cfg["n_generators"], out_dim=IN_DIM)
mq_final.load_state_dict(torch.load(f"{ASSET_DIR}/mediqgan_final.pt", map_location="cpu")["generator_state_dict"])
with torch.no_grad():
    final = mq_final(sample_z(N_EVAL)).numpy().reshape(N_EVAL, IMG_SIZE, IMG_SIZE)

rows = [("10 epoch(출발점)", gm.to_unit_scale(before, "tanh")),
        (f"+{FINE_TUNE_STEPS}step(오늘)", gm.to_unit_scale(after, "tanh")),
        ("25 epoch(강사 최종)", gm.to_unit_scale(final, "tanh"))]
fig, axes = plt.subplots(3, 4, figsize=(10, 7.5))
for r, (name, imgs) in enumerate(rows):
    for j in range(4):
        axes[r, j].imshow(imgs[j], cmap="gray"); axes[r, j].axis("off")
    axes[r, 0].set_ylabel(name)
    axes[r, 0].set_title(name, loc="left", fontsize=10)
plt.suptitle("MediQ-GAN - 학습이 진행되며 생성 결과가 어떻게 변하는가")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
gm.plot_loss_curves(ft_g, ft_d, ax=ax)
ax.set_title(f"오늘 돌린 미세조정 {FINE_TUNE_STEPS}step의 손실")
plt.tight_layout(); plt.show()

## 5. QGAN vs MediQ-GAN - 같은 지표로 비교 (20분)

두 모델은 출력 척도가 다름(QGAN은 patch별 합이 0 ~ 1인 확률, MediQ-GAN은 -1 ~ 1). 그대로 비교하면 척도 차이가 섞이므로
**둘 다 0 ~ 1로 되돌린 뒤** 같은 지표로 평가함.

생성 모델은 매번 새 난수를 쓰므로 지표가 실행마다 조금씩 달라짐. **소수점 자리보다 두 모델 사이의 격차를 볼 것.**

In [ ]:
table = gm.comparison_table({
    "QGAN (모델 단위 hybrid)": (qgan_fake, "prob"),
    "MediQ-GAN 10 epoch": (before, "tanh"),
    f"MediQ-GAN +{FINE_TUNE_STEPS}step": (after, "tanh"),
    "MediQ-GAN 25 epoch(최종)": (final, "tanh"),
}, df_imgs)
print("=== 생성품질 지표 비교 (모두 0~1로 되돌린 뒤 계산) ===")
display(table) if "display" in dir() else print(table)

In [ ]:
# 지표 ②③ 상세 - 어느 쪽이 DF의 형태를 더 잘 재현했는가
df_mean = gm.class_mean_image(real_u)  # DF의 전형적 형태
labels, corrs = [], []
for name, arr, mode in [("QGAN", qgan_fake, "prob"), ("MediQ-GAN 25 epoch", final, "tanh")]:
    u = gm.to_unit_scale(arr, mode)
    labels.append(name); corrs.append(gm.mean_image_correlation(u, df_mean).mean())

fig, axes = plt.subplots(1, 3, figsize=(12, 3.2))
axes[0].bar(labels, corrs); axes[0].axhline(0, color="gray", lw=0.8)
axes[0].set_title("DF 평균 이미지와의 상관계수"); axes[0].set_ylabel("상관계수")
gm.plot_pixel_histogram(real_u, gm.to_unit_scale(qgan_fake, "prob"), ax=axes[1],
                        label_real="실제 DF", label_fake="QGAN")
axes[1].set_title("QGAN 픽셀 분포")
gm.plot_pixel_histogram(real_u, gm.to_unit_scale(final, "tanh"), ax=axes[2],
                        label_real="실제 DF", label_fake="MediQ-GAN")
axes[2].set_title("MediQ-GAN 픽셀 분포")
plt.tight_layout(); plt.show()

for name, arr, mode in [("QGAN", qgan_fake, "prob"), ("MediQ-GAN 25 epoch", final, "tanh")]:
    u = gm.to_unit_scale(arr, mode)
    d, _ = gm.nearest_neighbor_distance(u, real_u)
    div = gm.sample_diversity(u)
    print(f"{name}: 최근접 실제거리 평균 {d.mean():.3f} (최솟값 {d.min():.3f}) / "
          f"샘플간 거리 평균 {div['pairwise_mean']:.3f}")
print("\n최근접거리가 0에 가까우면 암기, 샘플간 거리가 0에 가까우면 mode collapse임")

### MediQ-GAN 원 논문 실측 결과 (참고표 — 논문 수치 인용)

| 데이터셋 | 내용 | 결과 |
|---|---|---|
| ISIC 2019 | 피부경검사 8클래스 | 소수클래스(df, vasc, scc)에서 모든 고전 baseline보다 낮은 FID |
| ODIR-5k | 안저사진 | 증강 후 분류: 두 backbone(EfficientNet-B0, ViT-small) 모두 최고 성능 |
| RetinaMNIST | 다운샘플 안저사진 | 5개 클래스 중 4개에서 최저(최고품질) FID |

세 데이터셋 모두 클래스별 FID가 최고 또는 차상위임(논문 Table 2). ODIR-5k 행만 FID가 아니라 증강 후 분류 성능(Table 1)임.

비교 대상: DCGAN, WGAN-GP, Structure-Preserving GAN, StyleGAN2-ADA, FastGAN, **diffusion 모델(DiT, Stable Diffusion 3 - LoRA 미세조정)**,
그리고 이전 세대 hybrid 모델 MosaiQ까지 포함. IBM 실기기(`ibm_marrakesh`)에서 추론 검증도 완료함.

**시사점**: "생성자 전체를 양자로" 만드는 것보다, "고전이 잘하는 부분(전역 구조)은 고전에 맡기고
양자는 디테일 보강에만 사용하는" layer 단위 hybrid가 더 높은 해상도·더 좋은 품질로 확장 가능함을
보여줌 — 오늘 실습한 HAM10000 QGAN(생성자 전체 양자)의 자연스러운 발전 방향임.


## 6. 이미지 생성에서의 양자컴퓨팅

**① 측정 방식이 학습 성패를 가름**
ancilla qubit 없이 데이터 큐빗만 측정하면 patch의 밝기 총합이 1로 고정됨. segmentation 이미지는 배경 patch의 합이
0에 가까운데 이를 표현할 수 없어, 판별자가 그 차이만으로 항상 이기고 학습이 무너졌음(사전 실험).
ancilla qubit=0인 경우만 쓰는 방식으로 바꾸자 생성 이미지의 평균이 실제와 닮아졌음(8×8 단계 사전 실험, 생성 평균과 실제 평균의 상관계수 0.00 -> 0.49).
**회로를 바꾼 것이 아니라 측정·해석 방식을 바꾼 것**임.

**② 지금의 "양자 학습"은 대부분 실기가 아님**
오늘 두 모델 모두 exact 상태벡터 시뮬레이터로 학습했음(shots 미지정). 실기에서 shot 기반으로 학습하려면
parameter-shift rule이 필요하고, 기울기 계산에 파라미터당 회로 실행 2회가 듦(1일차 심화 참고).
MediQ-GAN 양자 파라미터 640개(5 × 8층 × 16큐빗)면 샘플 1개의 기울기에 1,280회(forward 5회 별도), batch 4면 step당 5,000회 이상임.

**③ 해상도 한계가 곧 평가 한계**
16x16에서는 FID 같은 표준 생성품질 지표가 의미를 갖지 못함(Inception 특징이 저해상도에서 무너짐).
그래서 오늘은 픽셀 통계·상관계수·최근접거리·다양성이라는 단순한 지표를 썼음. **"어떤 지표로 좋다고 할 것인가"가
생성모델 연구에서 아직 해결되지 않은 문제**라는 점을 그대로 보여줌.

**④ 큐빗 수가 늘면 시뮬레이션 비용이 지수로 늘어남**
MediQ-GAN의 16큐빗은 12큐빗 대비 상태 크기가 16배(2^16 vs 2^12)이고 실측 소요시간은 약 20배였음.
오늘 전체 학습 대신 미세조정을 한 이유가 이것임.

**⑤ 학습을 안정시키는 장치가 따로 필요함**
원 논문은 판별자를 WGAN-GP critic으로 둠 - gradient penalty로 판별자가 너무 세지지 않게 제약하는 장치임.
우리처럼 표준 BCE 판별자를 쓰면 판별자가 쉽게 이겨 생성자가 배울 신호를 잃음(사전학습 중 실제로 겪음, README "학습 안정성" 참고).
"양자 부분을 어떻게 설계하느냐"만큼이나 **"고전 쪽 학습을 어떻게 안정시키느냐"가 결과를 좌우함.**

**⑥ 그래서 무엇이 유효한가**
양자 생성모델은 양자 데이터·해밀토니안 상태 생성에 특화된 영역에서 먼저 의미를 가짐. 고전 데이터가 풍부하고
고전 모델이 잘하는 영역에서의 우위는 아직 검증 단계임. 오늘 본 것처럼 **"생성자 전체를 양자로"보다
"고전이 잘하는 전역 구조는 고전에 맡기고 양자는 일부만"**인 쪽이 더 높은 해상도로 확장되고 있음.

---
## 3일차 마무리 체크리스트
- [ ] 저장소에서 실습 자산 로드 완료
- [ ] QGAN 생성 + 지표 4종 확인
- [ ] MediQ-GAN 구조(dual-stream) 이해 - QGAN과의 hybrid 수준 차이 설명 가능
- [ ] MediQ-GAN 미세조정 실행 + 10 epoch/미세조정후/25 epoch 생성 결과 비교
- [ ] 두 모델 동일 지표 비교표 확인